# 🧬 DNABERT-2 + mCNN Pipeline for Transcription Factor Binding Site Prediction

This notebook implements a state-of-the-art deep learning model combining pre-trained `zhihan1996/DNABERT-2-117M` embeddings with a Multi-Scale Convolutional Neural Network (mCNN) to predict transcription factor binding sites (SP1, SP2, SP4, and Negative).

## Architecture Overview
1. **DNABERT-2 Token-Level Embeddings**: Extracts rich contextual representations of shape `(batch_size, sequence_length, 768)` from 101 bp sequences.
2. **Multi-Scale CNN (mCNN)**: Scans the embeddings using parallel 1D convolutions of kernel sizes `[3, 5, 7, 9]` to extract motif features at multiple widths, pools them to achieve translation invariance, and classifies them via fully connected layers.

### 1. Colab Setup and Installation
First, install `transformers` (Hugging Face) and `einops` (required for DNABERT-2 model code).

In [ ]:
# Install dependencies in Google Colab
!pip install -q transformers einops scikit-learn matplotlib seaborn torch

# Clone the repository (if running directly in a fresh Colab session)
# !git clone https://github.com/JustinYuanZe/SP1_TF_Biding_Project.git
# %cd SP1_TF_Biding_Project

### 2. Loading Modular Core Scripts
Import the custom PyTorch classes and training logic from our `src` folder.

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from IPython.display import Image, display

# Import modules from src/
from src.dnabert_wrapper import DNABERTWrapper
from src.mcnn_model import MultiScaleCNN
from src.train import DNAEmbeddingDataset, train_model, evaluate_model, plot_curves

### 3. Loading Dataset Sequences
Load the processed final dataset from `data/processed/`.

In [ ]:
sp1_path = os.path.join("data", "processed", "sp1_positive_final.fasta")
sp2_path = os.path.join("data", "processed", "sp2_positive_final.fasta")
sp4_path = os.path.join("data", "processed", "sp4_positive_final.fasta")
neg_path = os.path.join("data", "processed", "negative_final.fasta")

def load_fasta(path):
    seqs = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line.startswith('>'):
                seqs.append(line.upper())
    return seqs

print("Loading FASTA files...")
seqs_sp1 = load_fasta(sp1_path)
seqs_sp2 = load_fasta(sp2_path)
seqs_sp4 = load_fasta(sp4_path)
seqs_neg = load_fasta(neg_path)

sequences = seqs_sp1 + seqs_sp2 + seqs_sp4 + seqs_neg
y = np.concatenate([
    np.zeros(len(seqs_sp1)),
    np.ones(len(seqs_sp2)),
    np.full(len(seqs_sp4), 2),
    np.full(len(seqs_neg), 3)
], axis=0)

print(f"Total sequences: {len(sequences)}")
print(f"Labels distribution: {np.bincount(y.astype(int))}")

### 4. Embedding Extraction via DNABERT-2
We tokenize and extract the contextualized embeddings `(batch_size, 105, 768)` batch-by-batch using our `DNABERTWrapper`. Extracting them once saves training time during backpropagation epochs of the mCNN.

In [ ]:
# Set device and instantiate wrapper
device = "cuda" if torch.cuda.is_available() else "cpu"
wrapper = DNABERTWrapper(device=device)

# Extract embeddings (batch_size=64 by default)
print("\n--- Extracting DNA representations from DNABERT-2 ---")
embeddings = wrapper.get_embeddings(sequences)
print(f"Embeddings shape: {embeddings.shape}  # (samples, seq_len, 768)")

### 5. Stratified Split and PyTorch Dataloaders

In [ ]:
# Train/Validation Split (80% train, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(
    embeddings, y, test_size=0.2, stratify=y, random_state=42
)

# Create PyTorch datasets and dataloaders
train_dataset = DNAEmbeddingDataset(X_train, y_train)
val_dataset = DNAEmbeddingDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

print(f"Train dataloader batches: {len(train_loader)}")
print(f"Val dataloader batches:   {len(val_loader)}")

### 6. Training the Multi-Scale CNN (mCNN)

In [ ]:
# Initialize model
model = MultiScaleCNN(embedding_dim=768, branch_channels=128, num_classes=4, dropout_rate=0.5)

# Train the model for 15 epochs
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=15,
    lr=0.001,
    device=device,
    output_dir='models'
)

### 7. Performance Analysis and Plots
Let's plot the training convergence curves and evaluate our model's performance on the validation set using confusion matrix heatmaps and ROC/PR curves.

In [ ]:
# 1. Plot and save training convergence curves
plot_curves(history, save_dir='figures')
display(Image('figures/mcnn_training_curves.png'))

In [ ]:
# 2. Load the best saved model state
best_model_path = os.path.join("models", "best_mcnn_model.pt")
model.load_state_dict(torch.load(best_model_path, map_location=device))
print(f"Loaded best model weights from {best_model_path}")

# 3. Run detailed evaluation metrics (Classification Report, CM, ROC, PR Curves)
class_names = ['SP1', 'SP2', 'SP4', 'Negative']
evaluate_model(model, val_loader, class_names, device=device, save_dir='figures')

# Display Confusion Matrix and ROC Curves inline
print("\n--- Visualizing Performance Plots ---")
print("Confusion Matrix:")
display(Image('figures/confusion_matrix.png'))
print("ROC Curves:")
display(Image('figures/roc_curves.png'))
print("Precision-Recall Curves:")
display(Image('figures/precision_recall_curves.png'))